## Lab 1. MLP

In [20]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler
from sklearn.model_selection import train_test_split


#daecarga
houses = fetch_california_housing(as_frame=True)
df = houses.frame

### 1. Dataset

In [19]:
print("=== EXPLORACION Y PREPARACION DATASET ===")
print("\nDimensiones: ")
print(f" - Filas: {df.shape[0]}")
print(f" - Columnas: {df.shape[1]}")
print(f"\n{df.info()}")
print(f"\nEstadisticas:\n{df.describe()}")
print(f"\nVariable objetivo: {houses.target_names[0]}")
print(f"Variable feature: {list(houses.feature_names)}")

null_counts = df.isnull().sum()
print(f"\nValores nulos: {null_counts[null_counts > 0].to_dict() if any(null_counts > 0) else 'No hay valores nulos'}")
duplicates = df.duplicated().sum()
print(f"Valores duplicados: {duplicates}")

#outliers/atipicos con IQR
def detect_outliers_iqr(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]

    return {
        'col': col,
        'lower': lower,
        'upper': upper,
        'n_outliers': len(outliers),
        'total': len(outliers),
        'pct': len(outliers) / len(df) * 100
    }

print("Valores atipicos/variable:")
for col in df.select_dtypes(include=['float64', 'int64']).columns:
    if col != 'MedHouseVal':
        info = detect_outliers_iqr(df, col)
        if info['n_outliers'] > 0:
            print(f"   - {col}: {info['n_outliers']} outliers ({info['pct']:.1f}%)")
        else:
            print(f"   - {col}: No outliers")




=== EXPLORACION Y PREPARACION DATASET ===

Dimensiones: 
 - Filas: 20640
 - Columnas: 9
<class 'pandas.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   MedInc       20640 non-null  float64
 1   HouseAge     20640 non-null  float64
 2   AveRooms     20640 non-null  float64
 3   AveBedrms    20640 non-null  float64
 4   Population   20640 non-null  float64
 5   AveOccup     20640 non-null  float64
 6   Latitude     20640 non-null  float64
 7   Longitude    20640 non-null  float64
 8   MedHouseVal  20640 non-null  float64
dtypes: float64(9)
memory usage: 1.4 MB

None

Estadisticas:
             MedInc      HouseAge      AveRooms     AveBedrms    Population  \
count  20640.000000  20640.000000  20640.000000  20640.000000  20640.000000   
mean       3.870671     28.639486      5.429000      1.096675   1425.476744   
std        1.899822     12.585558      2.474173      0.

### 2. Exploracion y preparación de datos

In [21]:
def tratamiento_outlier(df, column):
    info = detect_outliers_iqr(df, column)
    n_outliers = info['total']
    pct_outliers = info['pct']
    
    print(f"\n--- {column} ---")
    print(f"Outliers: {n_outliers} ({pct_outliers:.1f}%)")
    print(f"Limite inferior: {info['lower']:.3f}")
    print(f"Limite superior: {info['upper']:.3f}")
    
    #valores extremos
    min_val = df[column].min()
    max_val = df[column].max()

    print(f"Rango: [{min_val:.2f}, {max_val:.2f}]")
    if pct_outliers == 0:
        rec = "No hay outliers detectados"
    elif pct_outliers < 1:
        rec = "Eliminar (pocos outliers, <1% del total)"
    elif pct_outliers < 3:
        rec = "Evaluar según modelo (1-3% del total)"
    elif pct_outliers < 5:
        rec = "Mantener o winsorizar (3-5% del total)"
    else:
        rec = "Mantener (más del 5%, es parte de la distribución)"

    return rec

for col in df.select_dtypes(include=['float64', 'int64']).columns:
    if col != 'MedHouseVal':
        tratamiento_outlier(df, col)

outliers_info = []
for col in df.select_dtypes(include=['float64', 'int64']).columns:
    if col != 'MedHouseVal':
        info = detect_outliers_iqr(df, col)
        outliers_info.append(info)

#prom de %
pct_outliers_total = np.mean([info['pct'] for info in outliers_info])
print(f"Porcentaje promedio de outliers: {pct_outliers_total:.1f}%")

if pct_outliers_total < 1:
    decision = "Eliminar atipicos (son pocos y no afectan significativamente)"
elif pct_outliers_total < 3:
    decision = "Mantener"
elif pct_outliers_total < 5:
    decision = "Mantener atipicos (son parte de la distribución real)"
else:
    decision = "Mantener atipicos (mas del 5% es que los datos pueden llegar a eso)"

print(f"\nDecision final: {decision}")


#escalar y separar datos 
X = df.drop('MedHouseVal', axis=1)
y = df['MedHouseVal']
# 70 entrenamiento, 15 validación, 15 prueba
X_train, X_temp, y_train, y_temp = train_test_split(X,y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print("Division dataset:")
print(f"\t - Entrenamiento: {len(X_train)} muestras")
print(f"\t - Validacion: {len(X_val)} muestras")
print(f"\t - Prueba: {len(X_test)} muestras")

scaler = RobustScaler()

print("Escalando datos...")
X_train_rob = scaler.fit_transform(X_train)

X_val_rob = scaler.transform(X_val)
X_test_rob = scaler.transform(X_test)

print(f"Scaler ajustado, {len(X_train)} muestras")
print(f"Validacion ({len(X_val)}) y prueba ({len(X_test)}) transformadas")



--- MedInc ---
Outliers: 681 (3.3%)
Limite inferior: -0.706
Limite superior: 8.013
Rango: [0.50, 15.00]

--- HouseAge ---
Outliers: 0 (0.0%)
Limite inferior: -10.500
Limite superior: 65.500
Rango: [1.00, 52.00]

--- AveRooms ---
Outliers: 511 (2.5%)
Limite inferior: 2.023
Limite superior: 8.470
Rango: [0.85, 141.91]

--- AveBedrms ---
Outliers: 1424 (6.9%)
Limite inferior: 0.866
Limite superior: 1.240
Rango: [0.33, 34.07]

--- Population ---
Outliers: 1196 (5.8%)
Limite inferior: -620.000
Limite superior: 3132.000
Rango: [3.00, 35682.00]

--- AveOccup ---
Outliers: 711 (3.4%)
Limite inferior: 1.151
Limite superior: 4.561
Rango: [0.69, 1243.33]

--- Latitude ---
Outliers: 0 (0.0%)
Limite inferior: 28.260
Limite superior: 43.380
Rango: [32.54, 41.95]

--- Longitude ---
Outliers: 0 (0.0%)
Limite inferior: -127.485
Limite superior: -112.325
Rango: [-124.35, -114.31]
Porcentaje promedio de outliers: 2.7%

Decision final: Mantener
Division dataset:
	 - Entrenamiento: 14448 muestras
	 - Vali

**¿Cuántas observaciones y cuántas variables tiene el dataset?** Total de servaciones y 9 variables\
**¿Qué representa cada variable (feature) y cuál es la variable objetivo (target)?**
La variable objetivo es MedHouseVal\
 Fatures:   
 - Longitude: que tan al oeste esta la casa (mientras mas alto el valor más al oeste)
 - Latitude: que tan al norte esta la casa (mientras mas alto el valor más al norte)
 - HouseAge: Edad de una casa en la cuadra
 - AveRooms: promedio de cuartos en la cuadra
 - AveBedrms: promedio de habitaciones en la cuadra
 - Population: total de poblacion en la cuadra
 - AveOccup: promedio de personas por casa
 - MedInc: Media de ingresos por casa por cuadra

**¿Hay valores nulos, duplicados o atípicos (outliers)? ¿Cómo los trató?**

Investigué un poco sobre esto y encontré que si eran menos del 5% total, se podian mantener

**¿Qué variables son numéricas y cuáles categóricas? ¿Cómo codificó las categóricas?**
- Numéricas: Las 9 variables son numéricas, especificamente float64
- Categóricas: Ninguna

**¿Fue necesario normalizar o escalar las variables numéricas?**
Debido a la gran diferencia de algunos rangos de valores, si, debido a que hay outliers robuscaler es mejor opcion ya uqe este es menos sensible a estos

### 3. Investigación: optimizadores y capas de PyTorch para el MLP

- nn. Linear: Aplica transformacion linear afin a datos de entrada  $y=xA^T+b$
- nn.ReLU: Aplica la funcion de unidad lineal rectificada por elemento  $ReLU(x) = (x)^+ = max(0,x)$
- nn.LeakyReLU: Aplica la funcion LeakyReLU por elemento  $LeakuReLU = max(0,x)+negative_slope∗min(0,x)$ o $\begin{cases} x &\text{if } x \ge 0 \\ negative Slope * X \text{, otherwise} \end{cases}$
- nn.Tanh: Aplica Tangente Hiperbólica por elemento  $Tanh(x)= \frac{exp(x)−exp(−x)}{exp(x)+exp(−x)}$
- nn.Dropout: Al entrenar algunos elemetos del input tensor se covnierten a 0 con proabilidad p. Estos elementos son elegidos de forma aleatoria e independiente de cada forward call.
- nn.BatchNorm1d: aplica la normalizacion por lotes a datos que son de una dimension (como vectores de series temporales)
- nn.MSELoss: Calcula error cuadrático (MSE) entre los valores reales y los que se predicen
- nn.L1Loss: Calcula el MAE (error absoluto medio) entre la salida y el valor real objetivo.
- nn.SmoothL1Loss: Funcion para calcular la perdida en aprendizaje automatico que combina las ventajas de L1 y L2 (MSE) usando un termino cuadratico para errores pequeños y lineal para grandes.
- torch.optim:
    - SGD: Implementa descenso de gradiente estocástico, actualiza pesos y sesgos de las capas al restar el gradiente a la funcion de perdida y multiplicando con la tasa de aprendizaje, con eso minimiza el error del modelo. La tasa de aprendizaje es fija para todos los pesos, no consume tanta memoria y deja añadir parametros opcionales. El parámetro lr controla el tamaño del paso del optimizador al actualizar pesos, weight_decay aplica una penalización con el fin de evitar un sobreajuste que pueda afectar los pesos.
    - Adam: Implementa el algoritmo Adam (Adaptive Moment Estimation), este es un algoritmo de optimización para entrenar modelos de Deep learning, combina el momento con tasas de aprendizaje adaptativas por cada parámetro con el fin de obtener una convergencia rápida y estable sin necesitar de ajustes complejos en los hiperparámetros. Ajusta los pesos y sesgos de cada capa de forma independiente después del cálculo de retropropagación, estabiliza el aprendizaje al usar promedios móviles exponenciales de los gradientes y puede funcionar de forma bastante efectiva con la tasa base.
    - RMSprop: Ajusta la tasa de aprendizaje para cada parámetro de forma adaptativa haciendo uso de los gradientes, hace los cálculos para determinar un promedio ponderado del cuadrado de los gradientes más recientes con el fin de escalar las actualizaciones de pesos y sesgos de cada capa. La tasa de aprendizaje se va adaptando a cada uno de los parámetros de forma individual, la inclusión de momento es opcional y permite que el gradiente se normalice por medio de una estimación de varianza. 


### 4. Entrenamiento e iteración de hiperparámetros